# SBD rewards for simulation posts

This notebook sums the final SBD rewards reported for the posts in `simulation_post_rewards.csv`. It uses `total_payout_value`, which represents the payout already given to a post. Pending payouts are displayed separately and are not included in the final total.

In [8]:
from pathlib import Path
import re

import pandas as pd


REWARDS_CSV = Path("simulation_post_rewards.csv")
rewards = pd.read_csv(REWARDS_CSV)


In [9]:
def parse_sbd(value: object) -> float:
    """Parse an API asset string such as '1.234 SBD' into its numeric amount."""
    if pd.isna(value) or not str(value).strip():
        return 0.0
    match = re.fullmatch(r"\s*([-+]?\d+(?:\.\d+)?)\s+SBD\s*", str(value))
    if not match:
        raise ValueError(f"Expected an SBD asset value, got {value!r}")
    return float(match.group(1))


required_columns = {"author", "title", "match_status", "total_payout_value"}
missing_columns = required_columns - set(rewards.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

rewards["total_payout_sbd"] = rewards["total_payout_value"].map(parse_sbd)
pending_values = rewards.get(
    "pending_payout_value", pd.Series(index=rewards.index, dtype=object)
)
rewards["pending_payout_sbd"] = pending_values.map(parse_sbd)

reward_view = rewards[[
    "author",
    "title",
    "match_status",
    "steem_url",
    "total_payout_value",
    "total_payout_sbd",
    "pending_payout_value",
    "pending_payout_sbd",
]].copy()
display(reward_view)


,author,title,match_status,steem_url,total_payout_value,total_payout_sbd,pending_payout_value,pending_payout_sbd
0,aibell,Daddykirbs Cool Curators Contest Entry - @aibell,matched,https://steemit.com/@aibell/daddykirbs-cool-cu...,0.216 SBD,0.216,0.000 SBD,0.0
1,aibell,How to Protect Your Plants From Frost,matched,https://steemit.com/@aibell/how-to-protect-you...,2.612 SBD,2.612,0.000 SBD,0.0
2,aibell,Awesome New Uses For Your Wood Ash From Your F...,matched,https://steemit.com/@aibell/awesome-new-uses-f...,2.656 SBD,2.656,0.000 SBD,0.0
3,aibell,The Drunk Dog -- A Scary Story With a Happy En...,matched,https://steemit.com/@aibell/the-drunk-dog-a-sc...,0.216 SBD,0.216,0.000 SBD,0.0
4,aibell,My 2018 Homesteading Resolutions,matched,https://steemit.com/@aibell/my-2018-homesteadi...,3.233 SBD,3.233,0.000 SBD,0.0
...,...,...,...,...,...,...,...,...
95,fawlingfeatherz,Vaux's Swift Playground,matched,https://steemit.com/@fawlingfeatherz/vaux-s-sw...,0.000 SBD,0.000,0.000 SBD,0.0
96,odin61,The spirit to live,matched,https://steemit.com/@odin61/the-spirit-to-live,0.000 SBD,0.000,0.000 SBD,0.0
97,flatjack,Dancing On Steven Hawkings Grave!!! AUTOHOAX S...,api_post_not_found,NaN,NaN,0.000,NaN,0.0
98,army-store24,tactical rifle scope 3-9x56EG with illuminated...,matched,https://steemit.com/@army-store24/tactical-rif...,0.000 SBD,0.000,0.000 SBD,0.0


In [10]:
paid_sbd = rewards["total_payout_sbd"].sum()
pending_sbd = rewards["pending_payout_sbd"].sum()
missing_payouts = rewards["total_payout_value"].fillna("").astype(str).str.strip().eq("").sum()
matched_posts = rewards["match_status"].eq("matched").sum()

summary = pd.DataFrame([{
    "posts_in_csv": len(rewards),
    "matched_posts": int(matched_posts),
    "posts_without_final_payout": int(missing_payouts),
    "total_final_reward_sbd": paid_sbd,
    "total_pending_payout_sbd_not_included": pending_sbd,
}])
display(summary)

print(f"Total SBD already given out: {paid_sbd:.3f} SBD")
if missing_payouts:
    print(
        f"Warning: {missing_payouts} row(s) have no total_payout_value. "
        "They contribute 0.000 SBD until the API lookup is completed."
    )


,posts_in_csv,matched_posts,posts_without_final_payout,total_final_reward_sbd,total_pending_payout_sbd_not_included
0,100,77,23,103.287,0.0


Total SBD already given out: 103.287 SBD
